## META | difficulty 1 | Objectives

This bundle introduces the transform itself in its negacyclic form.

Focus:

- the difference between `ω` and `ψ`
- direct NTTψ and INTTψ
- the direct convolution theorem in the negacyclic setting
- why this is still too slow at `O(n^2)` without butterflies


## META | difficulty 1 | Route Guardrails

You are at **step 7 of 27** in the only supported route.

Never choose the next notebook manually from the file tree. Use only the links in this cell and the final handoff cell.

**Immediate navigation**
- Next notebook: [Foundations / 02 Negative Wrapped NTT / Lab](lab.ipynb)
- Previous notebook: [Foundations / 01 Convolution To Toy NTT / Studio](../01_convolution_to_toy_ntt/studio.ipynb)
- Restart route: [Start Here](../../START_HERE.ipynb)

**Official route chain**
1. [Start Here](../../START_HERE.ipynb)
2. [Course Blueprint](../../COURSE_BLUEPRINT.ipynb)
3. [Foundations / 01 Convolution To Toy NTT / Lecture](../01_convolution_to_toy_ntt/lecture.ipynb)
4. [Foundations / 01 Convolution To Toy NTT / Lab](../01_convolution_to_toy_ntt/lab.ipynb)
5. [Foundations / 01 Convolution To Toy NTT / Problems](../01_convolution_to_toy_ntt/problems.ipynb)
6. [Foundations / 01 Convolution To Toy NTT / Studio](../01_convolution_to_toy_ntt/studio.ipynb)
7. **Foundations / 02 Negative Wrapped NTT / Lecture** <- you are here
8. [Foundations / 02 Negative Wrapped NTT / Lab](lab.ipynb)
9. [Foundations / 02 Negative Wrapped NTT / Problems](problems.ipynb)
10. [Foundations / 02 Negative Wrapped NTT / Studio](studio.ipynb)
11. [Butterfly Mechanics / 03 Fast Forward CT / Lecture](../../butterfly_mechanics/03_fast_forward_ct/lecture.ipynb)
12. [Butterfly Mechanics / 03 Fast Forward CT / Lab](../../butterfly_mechanics/03_fast_forward_ct/lab.ipynb)
13. [Butterfly Mechanics / 03 Fast Forward CT / Problems](../../butterfly_mechanics/03_fast_forward_ct/problems.ipynb)
14. [Butterfly Mechanics / 03 Fast Forward CT / Studio](../../butterfly_mechanics/03_fast_forward_ct/studio.ipynb)
15. [Butterfly Mechanics / 04 Fast Inverse GS / Lecture](../../butterfly_mechanics/04_fast_inverse_gs/lecture.ipynb)
16. [Butterfly Mechanics / 04 Fast Inverse GS / Lab](../../butterfly_mechanics/04_fast_inverse_gs/lab.ipynb)
17. [Butterfly Mechanics / 04 Fast Inverse GS / Problems](../../butterfly_mechanics/04_fast_inverse_gs/problems.ipynb)
18. [Butterfly Mechanics / 04 Fast Inverse GS / Studio](../../butterfly_mechanics/04_fast_inverse_gs/studio.ipynb)
19. [Kyber Mapping / 05 Kyber NTT And Base Multiplication / Lecture](../../kyber_mapping/05_kyber_ntt_and_base_multiplication/lecture.ipynb)
20. [Kyber Mapping / 05 Kyber NTT And Base Multiplication / Lab](../../kyber_mapping/05_kyber_ntt_and_base_multiplication/lab.ipynb)
21. [Kyber Mapping / 05 Kyber NTT And Base Multiplication / Problems](../../kyber_mapping/05_kyber_ntt_and_base_multiplication/problems.ipynb)
22. [Kyber Mapping / 05 Kyber NTT And Base Multiplication / Studio](../../kyber_mapping/05_kyber_ntt_and_base_multiplication/studio.ipynb)
23. [Professional / 06 Debugging NTT Failures / Lecture](../../professional/06_debugging_ntt_failures/lecture.ipynb)
24. [Professional / 06 Debugging NTT Failures / Lab](../../professional/06_debugging_ntt_failures/lab.ipynb)
25. [Professional / 06 Debugging NTT Failures / Problems](../../professional/06_debugging_ntt_failures/problems.ipynb)
26. [Professional / 06 Debugging NTT Failures / Studio](../../professional/06_debugging_ntt_failures/studio.ipynb)
27. [Course Complete](../../COURSE_COMPLETE.ipynb)


## MANDATORY | difficulty 2 | Why ψ Shows Up

For negative-wrapped convolution, the clean transform formula uses a `2n`-th root `ψ` with:

- `ψ^2 = ω`
- `ψ^n = -1`

That is what bakes the negacyclic sign rule into the transform itself.


In [ ]:
# MANDATORY | difficulty 2 | Inspect ω, ψ, And The Direct Transform Matrix

from IPython.display import display

from ntt_learning.toy_ntt import find_primitive_root, find_psi, ntt_psi_exponent_grid, ntt_psi_matrix
from ntt_learning.visuals import direct_ntt_player, plot_ntt_psi_exponent_heatmap, plot_ntt_psi_matrix_heatmap

modulus = 17
n = 4
omega = find_primitive_root(n, modulus)
psi = find_psi(n, modulus)

print("omega:", omega)
print("psi:", psi)
print("exponent grid:")
for row in ntt_psi_exponent_grid(n):
    print(row)
print("NTT_psi matrix:")
for row in ntt_psi_matrix(n, modulus, psi):
    print(row)

display(direct_ntt_player([1, 2, 3, 4], modulus, psi))
display(plot_ntt_psi_exponent_heatmap(n, title="Exponents 2ij + i for n=4"))
display(plot_ntt_psi_matrix_heatmap(n, modulus, psi, title="Concrete NTT_psi matrix in Z_17"))


## MANDATORY | difficulty 2 | Direct NTTψ Is Mechanically Clear But Still Quadratic

The direct transform is useful because every coefficient and every exponent is visible.
It is not yet efficient. It still performs the full matrix multiplication.


In [ ]:
# MANDATORY | difficulty 2 | Run A Direct NTTψ / INTTψ Round Trip

from IPython.display import display

from ntt_learning.toy_ntt import find_psi, forward_ntt_psi, inverse_ntt_psi
from ntt_learning.visuals import direct_ntt_player

signal = [1, 2, 3, 4]
modulus = 17
psi = find_psi(len(signal), modulus)
spectrum = forward_ntt_psi(signal, modulus, psi)

print("signal:", signal)
print("spectrum:", spectrum)
print("inverse recovery:", inverse_ntt_psi(spectrum, modulus, psi))
display(direct_ntt_player(signal, modulus, psi))


In [ ]:
# MANDATORY | difficulty 3 | Use Direct NTTψ For Negacyclic Multiplication

from IPython.display import display

from ntt_learning.toy_ntt import find_psi, forward_ntt_psi, inverse_ntt_psi, negacyclic_multiply, pointwise_multiply
from ntt_learning.visuals import plot_transform_pipeline

left = [1, 2, 3, 4]
right = [5, 6, 7, 8]
modulus = 17
psi = find_psi(4, modulus)

left_hat = forward_ntt_psi(left, modulus, psi)
right_hat = forward_ntt_psi(right, modulus, psi)
product_hat = pointwise_multiply(left_hat, right_hat, modulus)

print("NTT_psi(left):", left_hat)
print("NTT_psi(right):", right_hat)
print("pointwise product:", product_hat)
print("inverse of pointwise product:", inverse_ntt_psi(product_hat, modulus, psi))
print("schoolbook negacyclic:", negacyclic_multiply(left, right, n=4, modulus=modulus))
display(plot_transform_pipeline(left, right, modulus=modulus, psi=psi, title="Direct negacyclic multiply pipeline"))


## MANDATORY | difficulty 2 | Retrieval Check

1. Why is `ψ` stronger than `ω` in the negacyclic story?
2. What exact property does the inverse add that the forward transform does not?
3. Why are we still dissatisfied after seeing the direct transform work correctly?


In [ ]:
# FACULTATIVE | difficulty 4 | Optional: Compare Positive And Negative Wrapped Transforms

from ntt_learning.toy_ntt import find_primitive_root, find_psi, forward_ntt, forward_ntt_psi

signal = [1, 2, 3, 4]
modulus = 17
omega = find_primitive_root(4, modulus)
psi = find_psi(4, modulus)

print("positive-wrapped NTT:", forward_ntt(signal, modulus, omega))
print("negative-wrapped NTT_psi:", forward_ntt_psi(signal, modulus, psi))


## META | difficulty 1 | Next Notebook

You finished **Foundations / 02 Negative Wrapped NTT / Lecture**.

**Primary next action**
- Next notebook: [Step 8 of 27 - Foundations / 02 Negative Wrapped NTT / Lab](lab.ipynb)

**Recovery links if you get lost**
- Previous notebook: [Foundations / 01 Convolution To Toy NTT / Studio](../01_convolution_to_toy_ntt/studio.ipynb)
- Restart route: [Start Here](../../START_HERE.ipynb)
